In [1]:
import sklearn
import numpy as np
import keras_tuner
import tensorflow as tf
import matplotlib.pyplot as plt
import collections
import imblearn

import os, sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("Modules"))))

import Modules.ds_loader_new as ds_loader

dataset_loader = ds_loader.DatasetLoader(xlsx_path='../Data/Label_Map.xlsx', data_dir='../Data/ECGDataDenoised')
X, y = dataset_loader.load_data()

2025-04-18 21:46:19.176756: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-18 21:46:19.188544: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745005579.201637  131515 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745005579.206038  131515 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1745005579.217613  131515 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
X_train, X_temp, y_train, y_temp = sklearn.model_selection.train_test_split(
    X, y, test_size=0.3, random_state=42, shuffle=True
)

X_val, X_test, y_val, y_test = sklearn.model_selection.train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, shuffle=True
)
print("Unique classes in y:", np.unique(y_train))
print("Datatype:", (X_train.dtype), (y_train.dtype))
print(f"NaNs in X: {np.isnan(X_train).sum()}")
print(f"Infs in X: {np.isinf(X_train).sum()}")
print(f"Class distribution of training before SMOTE: {collections.Counter(y_train)}")
print(f"Class distribution of validation: {collections.Counter(y_val)}")
print(f"Class distribution of test: {collections.Counter(y_test)}")

print("\n[ ii ] Applying MinMax scaling to the dataset...")
scaler = sklearn.preprocessing.MinMaxScaler()

X_train_reshaped = X_train.reshape(-1, X_train.shape[-1])
X_val_reshaped = X_val.reshape(-1, X_val.shape[-1])
X_test_reshaped = X_test.reshape(-1, X_test.shape[-1])

print(
        f"Before scaling - X_train shape: {X_train.shape}, X_val shape: {X_val.shape}, X_test shape: {X_test.shape}"
)

X_train_reshaped = scaler.fit_transform(X_train_reshaped)
X_val_reshaped = scaler.transform(X_val_reshaped)
X_test_reshaped = scaler.transform(X_test_reshaped)

X_train = X_train_reshaped.reshape(X_train.shape[0], *X_train.shape[1:])
X_val = X_val_reshaped.reshape(X_val.shape[0], *X_val.shape[1:])
X_test = X_test_reshaped.reshape(X_test.shape[0], *X_test.shape[1:])

print(f"Min and Max of X_train: {np.min(X_train)}, {np.max(X_train)}")
print(f"Min and Max of X_val: {np.min(X_val)}, {np.max(X_val)}")
print(f"Min and Max of X_test: {np.min(X_test)}, {np.max(X_test)}")

print(f"\n\n[ ii ] Applying oversampling via SMOTE")

X_train_flat = X_train.reshape((X_train.shape[0], -1))
smote = imblearn.over_sampling.SMOTE(random_state=42)
X_resampled, y_train = smote.fit_resample(X_train_flat, y_train)
X_train = X_resampled.reshape((-1, *X_train.shape[1:]))
print(f"Class distribution after SMOTE: {collections.Counter(y_train)}")

Unique classes in y: [0 1 2 3]
Datatype: float32 int32
NaNs in X: 0
Infs in X: 0
Class distribution of training before SMOTE: Counter({np.int32(2): 2745, np.int32(1): 1582, np.int32(0): 1550, np.int32(3): 1534})
Class distribution of validation: Counter({np.int32(2): 565, np.int32(1): 348, np.int32(3): 347, np.int32(0): 328})
Class distribution of test: Counter({np.int32(2): 578, np.int32(3): 341, np.int32(0): 340, np.int32(1): 330})

[ ii ] Applying MinMax scaling to the dataset...
Before scaling - X_train shape: (7411, 500, 12), X_val shape: (1588, 500, 12), X_test shape: (1589, 500, 12)
Min and Max of X_train: 0.0, 1.0000001192092896
Min and Max of X_val: -0.0274541974067688, 1.282033920288086
Min and Max of X_test: 0.12314686179161072, 0.9169063568115234


[ ii ] Applying oversampling via SMOTE
Class distribution after SMOTE: Counter({np.int32(1): 2745, np.int32(3): 2745, np.int32(2): 2745, np.int32(0): 2745})


In [3]:
from rescnn import Resnet
model = Resnet()
model = model.build_model()
model.summary()

I0000 00:00:1745005626.254634  131515 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2463 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ ecg_sig             │ (None, 2500, 12)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ initial_conv        │ (None, 1250, 256) │     15,616 │ ecg_sig[0][0]     │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res_1_relu_1 (ReLU) │ (None, 1250, 256) │          0 │ initial_conv[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res_1_bn1           │ (None, 1250, 256) │      1,024 │ res_1_relu_1[0][… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res_1_conv_1        │ (None, 1250, 256) │    196,864 │ res_1_bn1[0][0]   │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res_1_relu_2 (ReLU) │ (None, 1250, 256) │          0 │ res_1_conv_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res_1_bn2           │ (None, 1250, 256) │      1,024 │ res_1_relu_2[0][… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res_1_conv_2        │ (None, 1250, 256) │    196,864 │ res_1_bn2[0][0]   │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res_1_relu_3 (ReLU) │ (None, 1250, 256) │          0 │ res_1_conv_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res_1_bn3           │ (None, 1250, 256) │      1,024 │ res_1_relu_3[0][… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 1250, 256) │          0 │ res_1_bn3[0][0],  │
│                     │                   │            │ initial_conv[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pool_1          │ (None, 623, 256)  │          0 │ add[0][0]         │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res_2_relu_1 (ReLU) │ (None, 623, 256)  │          0 │ max_pool_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res_2_bn1           │ (None, 623, 256)  │      1,024 │ res_2_relu_1[0][… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res_2_conv_1        │ (None, 623, 128)  │     98,432 │ res_2_bn1[0][0]   │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res_2_relu_2 (ReLU) │ (None, 623, 128)  │          0 │ res_2_conv_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res_2_bn2           │ (None, 623, 128)  │        512 │ res_2_relu_2[0][… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res_2_conv_2        │ (None, 623, 128)  │     49,280 │ res_2_bn2[0][0]   │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 645,764 (2.46 MB)

 Trainable params: 642,692 (2.45 MB)

 Non-trainable params: 3,072 (12.00 KB)

In [ ]:
RDIR="../src/Results/RES_500_00/" 
MDIR= RDIR + "RES_500_00.keras"
CDIR= RDIR + "C_RES_500_00.keras"
CVDIR = RDIR + "RES_500_00_CV.keras"


In [ ]:
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
    
early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
    
model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
        filepath=CDIR,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=130,
    batch_size=32,
    callbacks=[lr_scheduler,early_stopping,model_checkpoint]

)


In [ ]:

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

ax[0].plot(history.history['accuracy'], label='accuracy')
ax[0].plot(history.history['val_accuracy'], label='val_accuracy')
ax[0].set_title('Accuracy vs Val Accuracy')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Accuracy')
ax[0].legend()

ax[1].plot(history.history['loss'], label='loss')
ax[1].plot(history.history['val_loss'], label='val_loss')
ax[1].set_title('Loss vs Val Loss')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Loss')
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, batch_size=32)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

In [ ]:
y_pred = model.predict(X_test)

if y_pred.shape[1] == 1:  
    y_pred_binary = (y_pred > 0.5).astype(int)
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred)  
else:
    y_pred_binary = np.argmax(y_pred, axis=1)  
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred, multi_class='ovr')

print("Classification Report (Test Data):")
print(sklearn.metrics.classification_report(y_test, y_pred_binary))
print(f"AUC: {auc}")

y_train_pred = model.predict(X_train)
y_train_pred = np.argmax(y_train_pred, axis=1)

print("Classification Report (Train Data):")
print(sklearn.metrics.classification_report(y_train, y_train_pred))

In [ ]:
import seaborn as sns
y_pred_class = np.argmax(y_pred, axis=1)  
cm = sklearn.metrics.confusion_matrix(y_test, y_pred_class, normalize='true')

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
model.save(MDIR) 

In [ ]:
"""from rescnn import Resnet
RDIR="../src/Results/RES_500_00/" 
MDIR= RDIR + "RES_500_00.keras"
CDIR= RDIR + "C_RES_500_00.keras"
CVDIR = RDIR + "RES_500_00_CV.keras"

tuner = keras_tuner.Hyperband(
    Resnet(),
    objective='val_accuracy',
    max_epochs=120,
    overwrite=False,
    directory=RDIR,
    factor=3,
    project_name="RES_500_00",
)
tuner.search_space_summary()"""

In [ ]:
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
    
early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
    
model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
        filepath=CDIR,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
tuner.search(
    X_train, y_train, 
    epochs = 150,
    validation_data=(X_val, y_val),
    callbacks=[lr_scheduler,early_stopping,model_checkpoint]
)

In [ ]:
tuner.results_summary()

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()
best_model.save(MDIR) 

In [ ]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0] 
print(best_hps.values)

In [ ]:
test_loss, test_accuracy = best_model.evaluate(X_test, y_test, batch_size=32)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

In [ ]:
y_pred = best_model.predict(X_test)

if y_pred.shape[1] == 1:  
    y_pred_binary = (y_pred > 0.5).astype(int)
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred)  
else:
    y_pred_binary = np.argmax(y_pred, axis=1)  
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred, multi_class='ovr')

print("Classification Report (Test Data):")
print(sklearn.metrics.classification_report(y_test, y_pred_binary))
print(f"AUC: {auc}")

y_train_pred = best_model.predict(X_train)
y_train_pred = np.argmax(y_train_pred, axis=1)

print("Classification Report (Train Data):")
print(sklearn.metrics.classification_report(y_train, y_train_pred))

In [ ]:
import seaborn as sns
y_pred_class = np.argmax(y_pred, axis=1)  
cm = sklearn.metrics.confusion_matrix(y_test, y_pred_class, normalize='true')

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
kfold = sklearn.model_selection.KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
fold_histories = []

best_accuracy = 0.0
best_model = None  

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train, y_train)):
    print(f"\n--- Fold {fold+1} ---")

    X_tr, X_val_fold = X_train[train_idx], X_train[val_idx]
    y_tr, y_val_fold = y_train[train_idx], y_train[val_idx]

    model = Resnet().build(best_hps)

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val_fold, y_val_fold),
        epochs=100,
        callbacks=[lr_scheduler,early_stopping,model_checkpoint],
        verbose=1
    )

    val_loss, val_accuracy = model.evaluate(X_val_fold, y_val_fold, verbose=0)
    print(f"Fold {fold+1} Validation Accuracy: {val_accuracy:.4f}")
    fold_accuracies.append(val_accuracy)
    fold_histories.append(history)

    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_model = model
        model.save(CVDIR) 
        print(f"Saved best model from Fold {fold+1} with Accuracy: {val_accuracy:.4f}")

In [ ]:
print("Cross-validation accuracies:", fold_accuracies)
print("Average CV accuracy:", np.mean(fold_accuracies))
print("Max CV accuracy:", np.max(fold_accuracies))

In [ ]:
cv_model = tf.keras.models.load_model(CVDIR)
cv_model.evaluate(X_test, y_test)

In [ ]:
test_loss, test_accuracy = cv_model.evaluate(X_test, y_test, batch_size=32)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

In [ ]:
y_pred_probs = cv_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("Classification Report (Test Data):")
print(sklearn.metrics.classification_report(y_test, y_pred))

auc = sklearn.metrics.roc_auc_score(y_test, y_pred_probs, multi_class='ovr')
print(f"AUC (Test): {auc:.4f}")

y_train_probs = cv_model.predict(X_train)
y_train_pred = np.argmax(y_train_probs, axis=1)

print("Classification Report (Train Data):")
print(sklearn.metrics.classification_report(y_train, y_train_pred))

In [ ]:
import seaborn as sns

cm = sklearn.metrics.confusion_matrix(y_test, y_pred, normalize='true')

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

In [ ]:

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

ax[0].plot(history.history['accuracy'], label='accuracy')
ax[0].plot(history.history['val_accuracy'], label='val_accuracy')
ax[0].set_title('Accuracy vs Val Accuracy')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Accuracy')
ax[0].legend()

ax[1].plot(history.history['loss'], label='loss')
ax[1].plot(history.history['val_loss'], label='val_loss')
ax[1].set_title('Loss vs Val Loss')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Loss')
ax[1].legend()

plt.tight_layout()
plt.show()